#####REST API to Databrick - Custom Ingestion

In [0]:
import json
import requests
from datetime import datetime

url = "http://inceptezlabs.com/api.php"

'''
short version
api_resp = request.get(url)
print(api_resp.json)
data = api_resp.json()
proper_jsondata = json.dump(data)
ts = datetime.now().strftime("%Y%m%d%H%M%S")
out_path = f"/Volumes/lakeflow_pl_cat/lakeflow_pl_sch/cloud_datalake/apidata/post_{ts}.json"
dbutils.fs.put(out_path,proper_jsondata,overwrite=True)
'''
def fetch_save_data():
    try:
        api_resp = requests.get(url)
        if api_resp.status_code != 200:
            print(f"Failed to fetch data, status code: {api_resp.status_code}")
            print(f"Response: {api_resp.text}")
            return
        try:
            data = api_resp.json()
            print("Data recieved successfully")
            print(data)
        except json.JSONDecodeError:
            print("Error: repsonse is not valid json")
            return
    
        ts = datetime.now().strfdate("%Y%m%d%H%M%S")
        out_path = f"/Volumes/lakeflow_pl_cat/lakeflow_pl_sch/cloud_datalake/apidata/post_{ts}.json"

        try:
            dbutils.fs.put(
                out_path,
                json.dump(data),
                overwrite=True)
            print(f"successfully wrote data to output location:{out_path}")
        except NameError:
            print("Dbutils not defined, run in a databricks notebook")

    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    fetch_save_data()


#####Autoloader from Datalake to Bronze

In [0]:
'''
"uid": "308a1710-1233-46ec-800e-4bd3460bfe22",
        "user": {
            "name": "Afsheen Williams",
            "email": "afsheen@example.com",
            "location": "Paris",
            "registered": "2026-02-19T19:36:36+00:00"
'''

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

user_Schema = StructType([
    StructField("uid", StringType()),
    StructField("user", StructType([
        Structfield("name", StringType()),
        StructField("email", StringType()),
        StructField("location", StringType()),
        StructField("registered", StringType())
    ]))
])

df_raw = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.maxFilesPerTrigger", 1)
    .option("cloudFiles.schemaLocation","/Volumes/lakeflow_pl_cat/lakeflow_pl_sch/cloud_datalake/api_schemaL/_schema")
    .load("/Volumes/lakeflow_pl_cat/lakeflow_pl_sch/cloud_datalake/apidata/")
)

parsed_df = df_raw.withColumn("data", F.from_json(F.col("data"), user_schema))

df_user = (
    parsed_df.select(
        F.col("data.uid").alias("uid"),
        F.col("data.user.name").alias("user_name"),
        F.col("data.user.email").alias("user_email"),
        F.col("data.user.location").alias("user_location"),
        F.to_timestamp(F.col("data.user.registered")).alias("user_registered_ts")
        F.current_timestamp().alias("ingestion_ts")
    )
)
    
(df_user.writeStream
    .trigger(availableNow=True)
    .format("delta")
    .option("checkpointLocation", /Volumes/lakeflow_pl_cat/lakeflow_pl_sch/bronze/apidata_file/_checkpoint)
    .start("/Volumes/lakeflow_pl_cat/lakeflow_pl_sch/bronze/apidata")
        
)

(df_user.writeStream
    .trigger(availableNow=True)
    .format("delta")
    .option("checkpointLocation", /Volumes/lakeflow_pl_cat/lakeflow_pl_sch/silver/apidata_tab/_checkpoint)
    .toTable("lakeflow_pl_cat.lakeflow_pl_sch.bronze_user_apidata")
)

In [0]:
display(spark.sql("select * from lakeflow_pl_cat.lakeflow_pl_sch.bronze_user_apidata order by user_registered_ts desc"))

In [0]:
display(spark.read.format("delta").load("/Volumes/lakeflow_pl_cat/lakeflow_pl_sch/bronze/apidata").orderBy("user_registered_ts", ascending=False))
